In [1]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image


In [2]:
# Specify the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
import torch
from torch.nn.modules.linear import Identity 

# Register Identity as a safe global
torch.serialization.add_safe_globals([Identity])

# Ensure you match the model's original architecture
class ViTWithDropout(torch.nn.Module):
    def __init__(self, base_model, dropout_rate=0.5):
        super(ViTWithDropout, self).__init__()
        self.base_model = base_model
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        self.classifier = torch.nn.Linear(base_model.embed_dim, 2)

    def forward(self, x):
        x = self.base_model.forward_features(x)
        cls_token = x[:, 0, :]
        cls_token = self.dropout(cls_token)
        x = self.classifier(cls_token)
        return x

# Add ViTWithDropout if necessary
torch.serialization.add_safe_globals([ViTWithDropout])

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Explicitly set weights_only to False and map_location
model_path = "C:\\Users\\ezeki\\OneDrive\\Desktop\\MTechProject\\Models\\best_vit_model.pth"
loaded_model = torch.load(model_path, map_location=device, weights_only=False) 
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully on device:", device)


Model loaded successfully on device: cpu


In [5]:
# Print the architecture of the loaded model
print("Loaded Model Architecture:")
print(loaded_model)

Loaded Model Architecture:
ViTWithDropout(
  (base_model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
       

In [6]:
!pip install torchsummary



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from torchsummary import summary

# Ensure the model is on the appropriate device
loaded_model = loaded_model.to(device)

# Print a detailed summary of the model
print("Model Summary:")
summary(loaded_model, input_size=(3, 224, 224))  


Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 768, 14, 14]         590,592
          Identity-2             [-1, 196, 768]               0
        PatchEmbed-3             [-1, 196, 768]               0
           Dropout-4             [-1, 197, 768]               0
          Identity-5             [-1, 197, 768]               0
          Identity-6             [-1, 197, 768]               0
         LayerNorm-7             [-1, 197, 768]           1,536
            Linear-8            [-1, 197, 2304]       1,771,776
          Identity-9          [-1, 12, 197, 64]               0
         Identity-10          [-1, 12, 197, 64]               0
           Linear-11             [-1, 197, 768]         590,592
          Dropout-12             [-1, 197, 768]               0
        Attention-13             [-1, 197, 768]               0
         Identity-14    

In [11]:
from torchvision import transforms
from PIL import Image

preprocess = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), 
])

# Load an image
image_path = "C:\\Users\\ezeki\\OneDrive\\Desktop\\MTechProject\\eyepac-light-v2-512-jpg\\test\\NRG\\EyePACS-TRAIN-NRG-2928.jpg" 
image = Image.open(image_path)

# Apply preprocessing
input_tensor = preprocess(image).unsqueeze(0) 

In [12]:
# Define the class labels
class_labels = {0: "NRG", 1: "RG"}

# Perform inference
with torch.no_grad():
    output = loaded_model(input_tensor)

# Get the predicted class
_, predicted_class = torch.max(output, 1)
predicted_label = class_labels[predicted_class.item()]  

print(f"Predicted class: {predicted_label}")


Predicted class: NRG
